# Integrantes

Iago Neiva - RM 570234

Kelly Stephanie- RM 569044

Renan de Castro - RM 570532

In [ ]:
!apt-get update
!apt-get install -y zstd

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import subprocess
import time

# Encerra qualquer processo antigo ou órfão do ollama que possa ter travado
subprocess.run(["pkill", "ollama"])

# Inicia o servidor do Ollama em segundo plano
ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

# Aguarda 5 segundos para garantir que o servidor subiu completamente
time.sleep(5)
print("Ollama iniciado com sucesso e pronto para conexões!")

In [ ]:
!ollama pull llama3.2:1b

In [ ]:
import json
import requests
import sys

# 1. REGRAS DE ALERTA E PONTUAÇÃO DE RISCO

def avaliar_temperatura(t):
    if t < 18: return 'ATENÇÃO | Risco de congelamento', 1
    if 18 <= t < 30: return 'NORMAL | Temperatura estável', 0
    if 30 <= t < 35: return 'ATENÇÃO | Temperatura elevada', 1
    return 'CRÍTICO | Risco de superaquecimento', 2

def avaliar_comunicacao(c):
    if c < 30: return 'CRÍTICO | Comunicação com a base em nível crítico', 2
    if 30 <= c < 60: return 'ATENÇÃO | Comunicação instável', 1
    return 'NORMAL | Comunicação estável', 0

def avaliar_energia(b):
    if b < 20: return 'CRÍTICO | Bateria em nível crítico', 2
    if 20 <= b < 50: return 'ATENÇÃO | Bateria abaixo do recomendado', 1
    return 'NORMAL | Energia estável', 0

def avaliar_oxigenio(o):
    if o < 80: return 'CRÍTICO | Oxigênio em nível crítico', 2
    if 80 <= o < 90: return 'ATENÇÃO | Oxigênio abaixo do ideal', 1
    return 'NORMAL | Oxigênio adequado', 0

def avaliar_estabilidade(e):
    if e < 40: return 'CRÍTICO | Estabilidade operacional crítica', 2
    if 40 <= e < 70: return 'ATENÇÃO | Estabilidade operacional reduzida', 1
    return 'NORMAL | Estabilidade operacional adequada', 0

def obter_status_geral(pontos):
    if 0 <= pontos <= 2: return 'MISSÃO ESTÁVEL'
    if 3 <= pontos <= 7: return 'MISSÃO EM ATENÇÃO'
    return 'MISSÃO CRÍTICA'

# 2. FUNÇÕES DE INTEGRAÇÃO COM A IA

def consultar_ia_ciclo(status_geral, tendencia, dados_texto):
    payload = {
        "model": "llama3.2:1b",
        "temperature": 0.1,
        "messages": [
            {
                "role": "system",
                "content": "Você é o Mission Control AI. Com base nos dados resumidos fornecidos, gere apenas o 'Parecer Técnico' (máx 2 linhas) e 'Recomendações Rápidas' em tópicos curtos."
            },
            {
                "role": "user",
                "content": f"Status Geral: {status_geral}\nTendência: {tendencia}\nSistemas:\n{dados_texto}"
            }
        ],
        "stream": True
    }

    return processar_stream_ollama(payload)

def gerar_relatorio_final_ia(historico_pontos, chamados):
    # Lógica de veredicto em Python para aliviar a IA
    ultimo = historico_pontos[-1]
    if ultimo <= 2: veredicto = "SUCESSO"
    elif ultimo <= 7: veredicto = "SUCESSO PARCIAL"
    else: veredicto = "FALHA OPERACIONAL"

    payload = {
        "model": "llama3.2:1b",
        "temperature": 0.1,
        "messages": [
            {
                "role": "system",
                "content": "Você é o Mission Control AI. Escreva um sumário executivo final para o encerramento da missão com base nos dados do utilizador. Seja breve."
            },
            {
                "role": "user",
                "content": f"Veredito Técnico: {veredicto}\nChamados de IA efetuados: {chamados}\nGere um resumo executivo de fechamento focado nos gargalos operacionais."
            }
        ],
        "stream": True
    }

    return processar_stream_ollama(payload)

def processar_stream_ollama(payload):
    """Lê os pacotes de streaming do Ollama corretamente sem quebrar o JSON"""
    texto_gerado = ""
    try:
        response = requests.post("http://localhost:11434/api/chat", json=payload, stream=True)
        for line in response.iter_lines():
            if line:
                chunk = json.loads(line.decode('utf-8'))
                content = chunk.get("message", {}).get("content", "")
                print(content, end="", flush=True)
                texto_gerado += content
        print() # Quebra de linha no fim do bloco
    except Exception as err:
        print(f"\n[Erro na comunicação com o Ollama]: {err}")
    return texto_gerado

# 3. FLUXO PRINCIPAL DA MISSÃO

dados_missao = [
    [24, 92, 88, 96, 90],
    [27, 80, 72, 94, 85],
    [31, 65, 58, 91, 70],
    [36, 42, 38, 87, 55],
    [39, 28, 19, 78, 35],
    [34, 55, 32, 82, 50]
]

historico_risco = []
chamados_ia = 0

print('=' * 40)
print('         MISSION CONTROL AI  ')
print('=' * 40)
print('Missão: Orion Test Alpha\nEquipe: Equipe Apollo\nQuantidade de ciclos: 6')
print('=' * 40)

# Processamento Ciclo a Ciclo
for i, ciclo in enumerate(dados_missao):
    temp, com, bat, oxi, est = ciclo

    # Processamento analítico instantâneo em Python
    txt_temp, p_temp = avaliar_temperatura(temp)
    txt_com, p_com = avaliar_comunicacao(com)
    txt_bat, p_bat = avaliar_energia(bat)
    txt_oxi, p_oxi = avaliar_oxigenio(oxi)
    txt_est, p_est = avaliar_estabilidade(est)

    pontos_atuais = p_temp + p_com + p_bat + p_oxi + p_est
    historico_risco.append(pontos_atuais)

    status_geral = obter_status_geral(pontos_atuais)

    # Cálculo de tendência em Python
    if i == 0:
        tendencia = "A missão permaneceu estável"
    else:
        pontos_anteriores = historico_risco[i - 1]
        if pontos_atuais > pontos_anteriores:
            tendencia = "A missão apresentou tendência de piora"
        elif pontos_atuais < pontos_anteriores:
            tendencia = "A missão apresentou tendência de melhora"
        else:
            tendencia = "A missão permaneceu estável"

    # Relatório do ciclo
    print(f"\nStatus do ciclo {i+1}:")
    print(f"Temperatura: {temp}ºC | Comunicação: {com}% | Bateria: {bat}% | Oxigênio: {oxi}% | Estabilidade: {est}%")
    print(f"Diagnóstico Local: {status_geral} ({pontos_atuais} pts) | {tendencia}")

    # Interação com o Comandante
    while True:
        receber_analise = input('Deseja receber análise e recomendações da IA (sim | nao)? ').strip().lower()
        if receber_analise in ['sim', 's']:
            print("\n--- CONSULTANDO INTELIGÊNCIA ARTIFICIAL ---")
            dados_formatados = (
                f"- Temperatura: {temp}°C -> {txt_temp}\n"
                f"- Comunicação Base: {com}% -> {txt_com}\n"
                f"- Sistema de Energia: {bat}% -> {txt_bat}\n"
                f"- Suporte de Oxigênio: {oxi}% -> {txt_oxi}\n"
                f"- Estabilidade Operacional: {est}% -> {txt_est}"
            )
            consultar_ia_ciclo(status_geral, tendencia, dados_formatados)
            chamados_ia += 1
            break
        elif receber_analise in ['nao', 'não', 'n']:
            break
        else:
            print('Resposta inválida. Utilize "sim" ou "nao".')

# Finalização da Missão (Fim do Ciclo 6)
print('\n' + '=' * 50)
print('     GERANDO RELATÓRIO CONSOLIDADO FINAL')
print('=' * 50)
gerar_relatorio_final_ia(historico_risco, chamados_ia)
print(f"\n{'-' * 50}\nFim da simulação.")